# Reprodução de Identificação NARX (Sem `sysid`)
Implementação customizada do algoritmo OLS com ERR e simulação free-run.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures

# --- 1. Custom NARX Model (OLS with ERR) ---
class CustomNARX:
    def __init__(self, ny=5, nu=5, poly_order=2, n_components=10):
        self.ny = ny
        self.nu = nu
        self.poly_order = poly_order
        self.n_components = n_components
        self.p = max(ny, nu)
        self.poly = PolynomialFeatures(degree=poly_order, include_bias=False)
        self.selected_indices = []
        self.theta = []
        self.ERR = []
        self.feature_names = []
        
    def _extract_blocks(self, data_list):
        X, Y = [], []
        for (u, y) in data_list:
            N = len(y)
            for k in range(self.p, N):
                row = [y[k-i] for i in range(1, self.ny+1)] + [u[k-i] for i in range(1, self.nu+1)]
                X.append(row)
                Y.append(y[k])
        return np.array(X), np.array(Y)

    def fit(self, train_data):
        X_lin, y = self._extract_blocks(train_data)
        base_names = [f"y(k-{i})" for i in range(1, self.ny+1)] + [f"u(k-{i})" for i in range(1, self.nu+1)]
        X_poly = self.poly.fit_transform(X_lin)
        all_names = self.poly.get_feature_names_out(base_names)
        
        # Orthogonal Least Squares (OLS) with Error Reduction Ratio (ERR)
        M = X_poly.shape[1]
        N_samples = X_poly.shape[0]
        
        selected_indices = []
        ERR = []
        W = np.zeros((N_samples, self.n_components))
        A = np.zeros((self.n_components, self.n_components))
        g = np.zeros(self.n_components)
        
        var_y = np.dot(y, y)
        
        for i in range(self.n_components):
            best_err = -1
            best_idx = -1
            best_w = None
            best_a = None
            
            for j in range(M):
                if j in selected_indices: continue
                x_j = X_poly[:, j]
                a_j = np.zeros(i)
                w_j = x_j.copy()
                for k in range(i):
                    a_j[k] = np.dot(W[:, k], x_j) / (np.dot(W[:, k], W[:, k]) + 1e-12)
                    w_j -= a_j[k] * W[:, k]
                    
                norm_w_j = np.dot(w_j, w_j)
                if norm_w_j < 1e-12: continue
                
                g_j = np.dot(w_j, y) / norm_w_j
                err_j = (g_j**2 * norm_w_j) / var_y
                
                if err_j > best_err:
                    best_err = err_j
                    best_idx = j
                    best_w = w_j
                    best_a = a_j
                    
            selected_indices.append(best_idx)
            ERR.append(best_err)
            W[:, i] = best_w
            A[0:i, i] = best_a
            g[i] = np.dot(best_w, y) / np.dot(best_w, best_w)
            
        A_full = np.eye(self.n_components)
        for i in range(self.n_components):
            A_full[0:i, i] = A[0:i, i]
            
        self.theta = np.linalg.solve(A_full, g)
        self.selected_indices = selected_indices
        self.ERR = np.array(ERR)
        self.feature_names = [all_names[idx].replace(" ", "") for idx in selected_indices]
        
    def predict(self, u, y_history):
        N = len(u)
        yhat = np.zeros(N)
        yhat[:self.p] = y_history[:self.p]
        
        for k in range(self.p, N):
            row = [yhat[k-i] for i in range(1, self.ny+1)] + [u[k-i] for i in range(1, self.nu+1)]
            X_lin = np.array([row])
            X_poly = self.poly.transform(X_lin)
            X_sel = X_poly[0, self.selected_indices]
            yhat[k] = np.dot(X_sel, self.theta)
            yhat[k] = np.clip(yhat[k], -100.0, 100.0) # Clipping for stability
            
        return yhat

    def print_summary(self):
        print("=======================================================")
        print(" NARX model — selected terms and parameters (Custom OLS)")
        print("=======================================================")
        print(f"Max lag: {self.p} (ny={self.ny}, nu={self.nu}, l={self.poly_order})")
        print(f"{'#':<4} {'Term':<15} {'theta':<10} {'ERR (%)':<10}")
        print("-" * 55)
        for i in range(self.n_components):
            term = self.feature_names[i]
            th = self.theta[i]
            err_pct = self.ERR[i] * 100
            print(f"{i+1:<4} {term:<15} {th:>8.4f} {err_pct:>10.6f}")
        print("-" * 55)
        print(f"Total ERR explained: {np.sum(self.ERR)*100:.6f}%")

## 1. Load and preprocess the datasets
Corte dos sinais de 20 a 80 segundos com decimação (x5).

In [ ]:
DECIMATION = 5
T_START, T_END = 20.0, 80.0

def load_processed(name, t0=T_START, t1=T_END, decimation=DECIMATION):
    BASE2 = "https://raw.githubusercontent.com/FelipeEduardoMarcondes/SYSTEM-IDENTIFICATION-AERO/main/experimentos/RODADA-7/"
    mapping = {
        'multisine': 'multi-seno-60-050Hz_0904_20-38.csv',
        'swept_sine': 'chirp-60-amp50_0904_20-18.csv',
        'steps': 'seq-degraus-60-3_0904_20-57.csv'
    }
    df = pd.read_csv(BASE2 + mapping[name])
    
    t_full = df['tempo_ms'].values / 1000.0 if 'tempo_ms' in df.columns else np.arange(len(df)) * 0.01
    u_full = df['u_pct'].values if 'u_pct' in df.columns else df['motor_percent'].values
    y_full = df['angulo_deg'].values
    ref_full = df['referencia'].values if 'referencia' in df.columns else np.zeros_like(y_full)
    
    idx = np.where((t_full >= t0) & (t_full <= t1))[0]
    if len(idx) == 0: idx = np.arange(len(t_full))
    
    sl = slice(idx[0], idx[-1] + 1, decimation)
    return u_full[sl], y_full[sl], t_full[sl], ref_full[sl]

def plot_io(u, y, t, ref, title):
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 6), sharex=True)
    ax1.plot(t, ref, '--', color='red', label='Referência')
    ax1.plot(t, y, color='blue', label='Saída y')
    ax1.set_ylabel('Ângulo [°]'); ax1.set_title(title)
    ax1.legend(loc='upper right'); ax1.grid(True)
    
    ax2.plot(t, u, color='green', label='Controle u')
    ax2.set_xlabel('Tempo [s]'); ax2.set_ylabel('u')
    ax2.legend(loc='upper right'); ax2.grid(True)
    plt.tight_layout(); plt.show()

data = {
    'multisine': load_processed('multisine'),
    'swept_sine': load_processed('swept_sine'),
    'steps': load_processed('steps')
}

plot_io(*data['multisine'], 'Multisine')
plot_io(*data['swept_sine'], 'Swept sine')
plot_io(*data['steps'], 'Steps')

## 2. Train / test split
Divisão de 60/40 em 5 blocos intercalados.

In [ ]:
N_BLOCKS = 5
TRAIN_BLOCKS, TEST_BLOCKS = [0, 2, 4], [1, 3]

def block_bounds(M):
    return [int(round(M * i / N_BLOCKS)) for i in range(N_BLOCKS + 1)]

train_data, test_mask = [], {}
for name in data:
    u, y, t, ref = data[name]
    bnd = block_bounds(len(y))
    for b in range(N_BLOCKS):
        sl = slice(bnd[b], bnd[b+1])
        if b in TRAIN_BLOCKS:
            train_data.append((u[sl], y[sl]))
    
    m = np.zeros(len(y), bool)
    for b in TEST_BLOCKS:
        m[bnd[b]:bnd[b+1]] = True
    test_mask[name] = m

print(f"Training blocks: {len(train_data)} (60% of each of {len(data)} datasets)")

## 3. Model identification (multiple datasets)
Identificação conjunta dos blocos e extração de termos via OLS.

In [ ]:
ny_model = 5
nu_model = 5
poly_order_model = 2
n_components = 10

narx_model = CustomNARX(nu=nu_model, ny=ny_model, poly_order=poly_order_model, n_components=n_components)
narx_model.fit(train_data)
narx_model.print_summary()

## 4. Final evaluation
Free-run sobre todos os datasets (treino e teste intercalados).

In [ ]:
def free_run_full(name, title=None):
    u, y, t, ref = data[name]
    ml = narx_model.p
    y_fr = narx_model.predict(u, y_history=y[:ml])
    
    tt, ym = t[ml:], y[ml:]
    m = test_mask[name][ml:]
    rmse_test = np.sqrt(np.mean((ym[m] - y_fr[ml:][m]) ** 2))
    
    plt.figure(figsize=(12, 5))
    plt.plot(tt, ym, color='black', label='Measured y(k)')
    plt.plot(tt, y_fr[ml:], '--', color='crimson', label='NARX free-run')
    
    bnd = block_bounds(len(y))
    first = True
    for b in TEST_BLOCKS:
        a0 = max(bnd[b], ml)
        if a0 < bnd[b+1]:
            plt.axvspan(t[a0], t[bnd[b+1]-1], color='orange', alpha=0.15, label='Test region' if first else None)
            first = False
            
    plt.xlabel('Tempo [s]')
    plt.ylabel('Ângulo [°]')
    plt.title(f'{title or name} - full free-run (test RMSE = {rmse_test:.3f})')
    plt.legend(loc='upper right')
    plt.grid(True)
    plt.tight_layout()
    plt.show()
    return rmse_test

r1 = free_run_full('multisine', 'Multisine')
r2 = free_run_full('steps', 'Steps')
r3 = free_run_full('swept_sine', 'Swept sine')

print(f'\nFree-run test RMSE [deg] | multisine {r1:.3f} steps {r2:.3f} swept {r3:.3f}')